# TCPFormer on Fit3D: Inference and Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nestpr/fit3d-3d-pose-benchmark/blob/main/notebooks/01_tcpformer_inference.ipynb)

This notebook lifts 2D human poses to 3D with the pretrained **TCPFormer** model and evaluates the predictions against the motion-capture ground truth of the **Fit3D** dataset. It is part of the code accompanying the paper *"Comprehensive Evaluation of Monocular and Multi-view 3D Human Pose Estimation Models for Fitness Exercise Analysis"* ([arXiv:2611.02929](https://arxiv.org/abs/2611.02929)).

| | |
|---|---|
| **Model** | TCPFormer, Human3.6M checkpoint, 243-frame window |
| **Input** | 2D keypoints of camera 65906101 from one of three sources: ground-truth 2D (projection of the 3D ground truth), YOLO-Pose or RTMPose |
| **Output** | 3D predictions of shape `(T, 17, 3)` in metres, and per-sequence and per-joint error tables (MPJPE, PA-MPJPE) |
| **Experiment** | 224 sequences: 8 subjects × 28 exercises, warm-up sequences excluded |
| **Hardware** | GPU runtime (Tesla T4 in the paper), about 4 s per sequence |

**Contents:** 1. Requirements · 2. Environment setup · 3. Configuration · 4. Model loading · 5. Data · 6. Temporal inference · 7. Evaluation metrics · 8. Running the experiment · 9. Output verification · 10. Summary

## 1. Requirements

**Runtime.** A GPU runtime is required. The experiments of the paper were run in **Google Colab** (the reference environment) with Python 3.12, PyTorch 2.10, CUDA 12.8 and a Tesla T4. All inputs and outputs are stored under a single root folder, `STORAGE_ROOT`: Google Drive when the notebook runs in Colab, and the folder `data` next to the notebook otherwise.

**Running locally.** The notebook can also be run in Jupyter or VS Code on a machine with an NVIDIA GPU and `git`. The Drive mounting is skipped automatically, TCPFormer is cloned into the folder `external`, and `STORAGE_ROOT` (Section 3) must point to a directory with the layout below. Local execution has not been tested on all systems.

**Data.** Download Fit3D from <https://fit3d.imar.ro> (academic license). The 2D inputs are described in the repository README. The notebook expects:

```text
STORAGE_ROOT/
├── Fit3D_2D/<subject>/
│   ├── poses2d_gt/<exercise>.npy          # ground-truth 2D keypoints, (T, 17, 3)
│   ├── poses2d/<exercise>.npy             # YOLO-Pose keypoints, (T, 17, 3)  (+ optional <exercise>.json with width/height)
│   └── pose2d_rtmpose/<exercise>.json     # RTMPose detections (MMPose-style JSON, COCO-17)
├── VIDEOS/<subject>/<exercise>/joints3d_25.json   # Fit3D 3D ground truth, in metres
└── checkpoints/TCPFormer_h36m_243_379.pth.tr      # pretrained TCPFormer weights
```

The 3D ground truth is also accepted as `Fit3D_2D/<subject>/gt/<exercise>.json`, and the checkpoint is also searched in `Fit3D_eval/checkpoints/` and in `STORAGE_ROOT`. If a sequence has no `width`/`height` metadata, the native Fit3D resolution (900 × 900) is used and a warning is printed.

**Checkpoint.** The weights `TCPFormer_h36m_243_379.pth.tr` (Human3.6M, 243 frames) are released by the TCPFormer authors (<https://github.com/AsukaCamellia/TCPFormer>). Neither the TCPFormer code nor its weights are redistributed here; they keep their original license.

**How to run.** Execute the cells from top to bottom. To evaluate another 2D source, change `POSE_SOURCE` in Section 3 and run again from that cell downwards; every source writes to its own files. Sequences that were already processed are skipped, so an interrupted run can be resumed.

## 2. Environment Setup

The official TCPFormer repository is cloned into the working directory (`/content` in Colab, `external` locally) and the required Python packages are installed. TCPFormer is treated as an external dependency: the code is fetched at run time, while the data, the checkpoint and all results live in `STORAGE_ROOT`.

> **Note.** In Google Colab, the first execution of the installation cell may print several lines labelled `ERROR: pip's dependency resolver ...`. They are dependency-conflict notices, not failures: the pinned `protobuf==4.25.3` differs from the version required by libraries preinstalled in Colab (TensorBoard, TensorFlow, Google Cloud client libraries), none of which is used by this notebook. The cell ends with the message *"TCPFormer and required dependencies installed successfully."*, and the notices do not appear again if the cell is executed a second time.

In [ ]:
# Detect the runtime and prepare the working directory

import os
import shutil
import sys

try:
    import google.colab   # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Directory of the notebook (the working directory when the notebook starts). It is kept if this cell is run
# again, because later cells change the working directory.
PROJECT_DIR = globals().get("PROJECT_DIR") or os.getcwd()

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_DIR = "/content"                        # third-party code is cloned here
else:
    WORK_DIR = os.path.join(PROJECT_DIR, "external")   # third-party code is cloned here
    os.makedirs(WORK_DIR, exist_ok=True)

TCPFORMER_ROOT = os.path.join(WORK_DIR, "TCPFormer")

print(f"Runtime: {'Google Colab' if IN_COLAB else 'local'} | working directory: {WORK_DIR}")

In [ ]:
# Clone TCPFormer and install the dependencies

shutil.rmtree(TCPFORMER_ROOT, ignore_errors=True)

%cd {WORK_DIR}
!git clone https://github.com/AsukaCamellia/TCPFormer.git

# Revision of the cloned repository. It is printed so that it can be recorded; set REPO_COMMIT to a commit hash
# to reproduce exactly that revision of the original code.
REPO_COMMIT = None

import subprocess

if REPO_COMMIT:
    subprocess.run(["git", "-C", TCPFORMER_ROOT, "checkout", "-q", REPO_COMMIT], check=True)

commit = subprocess.run(["git", "-C", TCPFORMER_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("TCPFormer repository commit:", commit)

%cd {TCPFORMER_ROOT}

!{sys.executable} -m pip install -q \
    protobuf==4.25.3 \
    opencv-python \
    imageio \
    imageio-ffmpeg \
    easydict \
    pyyaml \
    tqdm

print("TCPFormer and required dependencies installed successfully.")

In [ ]:
# Imports

import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
import torch
import yaml

## 3. Configuration

All user-editable settings are defined in the next cell; the rest of the notebook only reads them.

| Setting | Meaning |
|---|---|
| `STORAGE_ROOT` | Root folder of all inputs and outputs: Google Drive in Colab, the folder `data` otherwise. Edit it for a local run. |
| `POSE_SOURCE` | 2D input to evaluate: `"pose2d_rtmpose"` (main setting of the paper), `"poses2d_gt"` or `"pose2d_yolo"`. |
| `CLIP_LEN` | Window length in frames, fixed to 243 by the pretrained model. |
| `OVERLAP_STRIDE` | Distance between the starts of consecutive windows. The windows overlap by `CLIP_LEN - OVERLAP_STRIDE` = 183 frames. |
| `OVERLAP_BATCH_SIZE` | Windows processed simultaneously on the GPU. It only affects speed and memory, not the results. |
| `EXCLUDE_WARMUPS` | Exclude the warm-up sequences, as in the paper. |
| `SUBJECTS`, `MAX_SEQUENCES_PER_SUBJECT` | Optional subset for a quick test. With the default `None` the complete experiment (224 sequences) is run. A subset is written to a separate output folder. |
| `DEFAULT_IMAGE_SIZE` | Image size assumed when a sequence has no `width`/`height` metadata. |

The remaining lines of the cell derive the input and output locations. The pretrained checkpoint is searched in several standard locations and the first one found is used.

In [ ]:
# ----------------------------------------------------------------------------
# User-editable settings
# ----------------------------------------------------------------------------
STORAGE_ROOT = "/content/drive/MyDrive" if IN_COLAB else os.path.join(PROJECT_DIR, "data")   # root folder of all inputs and outputs

POSE_SOURCE = "pose2d_rtmpose"            # "pose2d_rtmpose", "poses2d_gt" or "pose2d_yolo"

CLIP_LEN = 243                            # window length of the pretrained model (frames)
OVERLAP_STRIDE = 60                       # frames between consecutive window starts
OVERLAP_BATCH_SIZE = 8                    # windows per forward pass (reduce if out of memory)

EXCLUDE_WARMUPS = True                    # exclude warm-up sequences
SUBJECTS = None                           # e.g. ["s03"] for a quick test; None = all subjects
MAX_SEQUENCES_PER_SUBJECT = None          # e.g. 3 for a quick test; None = all sequences

DEFAULT_IMAGE_SIZE = {"width": 900, "height": 900}   # Fit3D native resolution

# ----------------------------------------------------------------------------
# Derived locations
# ----------------------------------------------------------------------------
ROOT_2D = os.path.join(STORAGE_ROOT, "Fit3D_2D")      # 2D pose sources
VIDEOS_ROOT = os.path.join(STORAGE_ROOT, "VIDEOS")    # Fit3D 3D ground truth

IS_SUBSET = SUBJECTS is not None or MAX_SEQUENCES_PER_SUBJECT is not None
SAVE_ROOT = os.path.join(STORAGE_ROOT, "Fit3D_TCPFormer" + ("_subset" if IS_SUBSET else ""))

CHECKPOINT_NAME = "TCPFormer_h36m_243_379.pth.tr"
CHECKPOINT_CANDIDATES = [
    os.path.join(STORAGE_ROOT, "checkpoints", CHECKPOINT_NAME),
    os.path.join(STORAGE_ROOT, "Fit3D_eval", "checkpoints", CHECKPOINT_NAME),
    os.path.join(STORAGE_ROOT, CHECKPOINT_NAME),
]
CHECKPOINT_PATH = next(
    (path for path in CHECKPOINT_CANDIDATES if os.path.isfile(path)),
    CHECKPOINT_CANDIDATES[0],
)

RUN_TAG = f"overlap_hann_w{CLIP_LEN}_s{OVERLAP_STRIDE}"

assert 0 < OVERLAP_STRIDE <= CLIP_LEN, "OVERLAP_STRIDE must be in the range (0, CLIP_LEN]."

# ----------------------------------------------------------------------------
# Runtime
# ----------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.backends.cudnn.benchmark = True

os.makedirs(SAVE_ROOT, exist_ok=True)

print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | CUDA {torch.version.cuda} | device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"2D pose source: {POSE_SOURCE}")
print(f"Windows: {CLIP_LEN} frames, stride {OVERLAP_STRIDE} (overlap {CLIP_LEN - OVERLAP_STRIDE}), batch {OVERLAP_BATCH_SIZE}")
print(f"2D inputs:  {ROOT_2D}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Outputs:    {SAVE_ROOT}" + ("   (subset run)" if IS_SUBSET else ""))

## 4. Model Loading

TCPFormer is instantiated from the **Human3.6M 243-frame configuration** of the original repository and the pretrained checkpoint is loaded. Checkpoint keys prefixed with `module.` (weights saved from multi-GPU training) are renamed before loading.

The numbers of missing and unexpected keys are reported as a consistency check: the checkpoint must match the architecture exactly, i.e., zero missing and zero unexpected keys. The model is then switched to evaluation mode.

In [ ]:
# Load the pretrained TCPFormer model

sys.path.insert(0, TCPFORMER_ROOT)

from utils.learning import load_model_TCPFormer


class AttrDict(dict):
    """Dictionary allowing attribute-style access, as expected by the TCPFormer loader."""

    def __getattr__(self, name):
        try:
            value = self[name]
        except KeyError:
            raise AttributeError(name) from None

        return AttrDict(value) if isinstance(value, dict) else value


if not os.path.isfile(CHECKPOINT_PATH):
    raise FileNotFoundError(
        "TCPFormer checkpoint not found. Place it in one of these locations:\n  "
        + "\n  ".join(CHECKPOINT_CANDIDATES)
    )

CONFIG_PATH = os.path.join(TCPFORMER_ROOT, "configs", "h36m", "TCPFormer_h36m_243.yaml")

with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = AttrDict(yaml.safe_load(file))

model = load_model_TCPFormer(config).to(device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
state_dict = {key.replace("module.", ""): value for key, value in state_dict.items()}

missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {len(missing_keys)} | unexpected keys: {len(unexpected_keys)}")

if missing_keys or unexpected_keys:
    for key in (missing_keys + unexpected_keys)[:30]:
        print(f"  {key}")

    raise RuntimeError(
        "The checkpoint does not exactly match the TCPFormer architecture. "
        "Inference has been stopped to avoid using partially loaded weights."
    )

model.eval()

print(f"Configuration: {CONFIG_PATH}")

## 5. Data

### 5.1 Joint layout and 2D input conventions

TCPFormer works with the 17-joint Human3.6M-style skeleton:

$$
[\text{Pelvis}, \text{RHip}, \text{RKnee}, \text{RAnkle},
\text{LHip}, \text{LKnee}, \text{LAnkle},
\text{Spine}, \text{Neck}, \text{Nose}, \text{Head},
\text{LShoulder}, \text{LElbow}, \text{LWrist},
\text{RShoulder}, \text{RElbow}, \text{RWrist}]
$$

Each 2D sequence has shape $(T, 17, 3)$, where the last dimension holds $x$, $y$ and the keypoint confidence (a score of 1 is assigned when it is not available).

- **RTMPose** detections follow the COCO-17 layout and are converted when they are loaded. Pelvis and Neck are the midpoints of the two hips and of the two shoulders, Spine is the midpoint between Pelvis and Neck, and Nose and Head are both set to the nose keypoint of COCO-17 (there is no head-top keypoint), so the COMMON15 skeleton of the paper uses the Nose joint (index 9) and discards Spine (7) and Head (10).
- **YOLO-Pose** keypoints are stored directly in the 17-joint layout (the same conversion is applied when they are extracted).
- **Ground-truth 2D** keypoints are the projection of the Fit3D 3D ground truth. Their left/right leg indexing is the opposite of the one expected by the model, so the hip, knee and ankle joints are exchanged (indices $[1,2,3] \leftrightarrow [4,5,6]$) before inference. This correction is not needed for YOLO-Pose and RTMPose.

Finally, the image coordinates are normalized with the frame width $W$ and height $H$ following the TCPFormer convention:

$$
x' = \frac{2x}{W} - 1, \qquad y' = \frac{2y}{W} - \frac{H}{W}
$$

In [ ]:
# Joint layout, 2D conversions and normalization

JOINT_NAMES = [
    "Pelvis", "RHip", "RKnee", "RAnkle",
    "LHip", "LKnee", "LAnkle",
    "Spine", "Neck", "Nose", "Head",
    "LShoulder", "LElbow", "LWrist",
    "RShoulder", "RElbow", "RWrist",
]

# COCO-17 keypoints are extended with three derived joints (17: Pelvis, 18: Neck,
# 19: Spine); this list selects, for every Human3.6M joint, its source index.
H36M_FROM_EXTENDED = [17, 12, 14, 16, 11, 13, 15, 19, 18, 0, 0, 5, 7, 9, 6, 8, 10]

LEG_SWAP = [4, 5, 6, 1, 2, 3]   # RHip, RKnee, RAnkle <-> LHip, LKnee, LAnkle


def coco17_to_h36m17(coco):
    """Convert COCO-17 keypoints (T, 17, C) to the 17-joint layout of TCPFormer."""
    coco = np.asarray(coco, dtype=np.float32)

    pelvis = (coco[:, 11] + coco[:, 12]) / 2.0
    neck = (coco[:, 5] + coco[:, 6]) / 2.0
    spine = (pelvis + neck) / 2.0

    extended = np.concatenate(
        [coco, pelvis[:, None], neck[:, None], spine[:, None]], axis=1
    )

    return extended[:, H36M_FROM_EXTENDED].astype(np.float32)


def swap_legs(poses):
    """Exchange the left and right leg joints of a (T, 17, C) sequence."""
    output = poses.copy()
    output[:, 1:7] = poses[:, LEG_SWAP]
    return output


def ensure_3_channels(keypoints):
    """Return the keypoints as (T, 17, 3), adding a confidence of 1 if it is missing."""
    keypoints = np.asarray(keypoints, dtype=np.float32)

    if keypoints.ndim != 3 or keypoints.shape[1] != 17 or keypoints.shape[2] not in (2, 3):
        raise ValueError(f"Expected shape (T, 17, 2 or 3), received {keypoints.shape}.")

    if keypoints.shape[2] == 2:
        confidence = np.ones(keypoints.shape[:2] + (1,), dtype=np.float32)
        keypoints = np.concatenate([keypoints, confidence], axis=-1)

    return keypoints


def normalize_keypoints(keypoints, width, height):
    """Normalize image coordinates following the TCPFormer input convention."""
    output = np.asarray(keypoints, dtype=np.float32).copy()

    output[..., 0] = output[..., 0] / width * 2.0 - 1.0
    output[..., 1] = output[..., 1] / width * 2.0 - (height / width)

    return output

### 5.2 Loading the 2D pose sources

The three sources are stored differently, so each one has its own loader. They all return a sequence of shape $(T, 17, 3)$ together with the image size used for normalization.

- **Ground-truth 2D and YOLO-Pose** are NumPy arrays, one `.npy` file per sequence. The image size is read from the `.json` file with the same name when it exists, or from the metadata of the other sources of the same sequence.
- **RTMPose** detections are MMPose-style JSON files: a list of frames, each with a `frame_id` and its `instances` (`keypoints`, `keypoint_scores`, `bbox_score`). For every frame the instance with the highest `bbox_score` is used. A frame without detection repeats the most recent valid pose, and frames before the first detection use the first valid pose. The JSON files carry no image size, which is therefore taken from the metadata of the other sources.

The folder names accepted for every source, and whether the leg correction applies, are defined in the `POSE_SOURCES` table.

In [ ]:
# 2D pose sources: folders, formats and loaders

POSE_SOURCES = {
    "poses2d_gt": {
        "folders": ["poses2d_gt", "pose2d_gt", "poses2d_oracle", "pose2d_oracle"],
        "format": "npy",
        "swap_legs": True,
    },
    "pose2d_yolo": {
        "folders": ["pose2d_yolo", "poses2d_yolo", "poses2d", "pose2d"],
        "format": "npy",
        "swap_legs": False,
    },
    "pose2d_rtmpose": {
        "folders": ["pose2d_rtmpose", "poses2d_rtmpose"],
        "format": "json",
        "swap_legs": False,
    },
}

if POSE_SOURCE not in POSE_SOURCES:
    raise ValueError(f"POSE_SOURCE must be one of {list(POSE_SOURCES)}, received '{POSE_SOURCE}'.")

SOURCE = POSE_SOURCES[POSE_SOURCE]

# Folders that may contain the width/height metadata of a sequence.
METADATA_FOLDERS = POSE_SOURCES["pose2d_yolo"]["folders"] + POSE_SOURCES["poses2d_gt"]["folders"]

_size_warnings = set()


def is_warmup(video_name):
    """Warm-up sequences are excluded from the experiments."""
    return str(video_name).lower().startswith("warmup")


def resolve_pose_dir(subject):
    """Return the folder of the selected 2D source for a subject, or None."""
    for folder in SOURCE["folders"]:
        candidate = os.path.join(ROOT_2D, subject, folder)

        if os.path.isdir(candidate):
            return candidate

    return None


def list_pose_videos(pose_dir):
    """List the sequences available for the selected 2D source."""
    extension = "." + SOURCE["format"]

    videos = sorted(
        name[:-len(extension)]
        for name in os.listdir(pose_dir)
        if name.endswith(extension) and not name.endswith("_gt3d_cam.npy")
    )

    if EXCLUDE_WARMUPS:
        videos = [video for video in videos if not is_warmup(video)]

    return videos


def load_image_size(subject, video_name, preferred_dir=None):
    """Return {"width", "height"} of a sequence from the available metadata files."""
    folders = ([preferred_dir] if preferred_dir else []) + [
        os.path.join(ROOT_2D, subject, folder) for folder in METADATA_FOLDERS
    ]

    for folder in folders:
        path = os.path.join(folder, f"{video_name}.json")

        if not os.path.exists(path):
            continue

        try:
            with open(path, "r", encoding="utf-8") as file:
                metadata = json.load(file)
        except Exception:
            continue

        if isinstance(metadata, dict) and "width" in metadata and "height" in metadata:
            return metadata

    if (subject, video_name) not in _size_warnings:
        _size_warnings.add((subject, video_name))
        print(
            f"WARNING | {subject}/{video_name} | no width/height metadata found; "
            f"using the default image size {DEFAULT_IMAGE_SIZE['width']}x{DEFAULT_IMAGE_SIZE['height']}."
        )

    return dict(DEFAULT_IMAGE_SIZE)


def load_rtmpose_json(path):
    """Load an RTMPose JSON file as a (T, 17, 3) sequence in the TCPFormer layout."""
    with open(path, "r", encoding="utf-8") as file:
        frames = json.load(file)

    if not isinstance(frames, list) or len(frames) == 0:
        raise ValueError(f"RTMPose JSON must contain a non-empty list: {path}")

    num_frames = max(int(frame.get("frame_id", index)) for index, frame in enumerate(frames)) + 1
    coco = np.full((num_frames, 17, 3), np.nan, dtype=np.float32)   # x, y, score

    for index, frame in enumerate(frames):
        frame_id = int(frame.get("frame_id", index))
        instances = frame.get("instances", [])

        if len(instances) == 0:
            continue

        best = max(instances, key=lambda instance: float(instance.get("bbox_score", 0.0)))
        keypoints = np.asarray(best["keypoints"], dtype=np.float32)

        if keypoints.shape[0] < 17:
            raise ValueError(f"RTMPose frame {frame_id} contains fewer than 17 keypoints.")

        scores = np.asarray(best.get("keypoint_scores", np.ones(17)), dtype=np.float32)

        if scores.shape[0] < 17:
            scores = np.ones(17, dtype=np.float32)

        coco[frame_id, :, :2] = keypoints[:17, :2]
        coco[frame_id, :, 2] = scores[:17]

    valid = ~np.isnan(coco).any(axis=(1, 2))

    if not valid.any():
        raise ValueError(f"No valid RTMPose detections found in {path}.")

    # Fill gaps with the most recent valid frame (the first valid one before it).
    last_valid = np.maximum.accumulate(np.where(valid, np.arange(num_frames), -1))
    last_valid[last_valid < 0] = np.flatnonzero(valid)[0]

    return coco17_to_h36m17(coco[last_valid])


def load_2d_keypoints(subject, video_name, pose_dir):
    """Load a 2D sequence (T, 17, 3) and the image size of the selected source."""
    path = os.path.join(pose_dir, f"{video_name}.{SOURCE['format']}")

    if not os.path.exists(path):
        raise FileNotFoundError(f"2D keypoint file not found: {path}")

    if SOURCE["format"] == "json":
        return load_rtmpose_json(path), load_image_size(subject, video_name)

    keypoints = ensure_3_channels(np.load(path))

    return keypoints, load_image_size(subject, video_name, preferred_dir=pose_dir)

### 5.3 Fit3D ground truth

Fit3D provides the 3D annotations in the `joints3d_25` field (25 joints, in metres). For this evaluation the first 17 joints are kept and the sequence is root-aligned by subtracting the pelvis in every frame. A fixed left/right leg exchange is applied to the ground truth to match the joint convention of the model; the output of TCPFormer is **not** reordered.

In [ ]:
# Ground-truth utilities

def root_align(poses):
    """Root-align a (T, J, 3) sequence using the pelvis (joint 0)."""
    return poses - poses[:, 0:1, :]


def find_gt_path(subject, video_name):
    """Locate the Fit3D 3D ground-truth file of a sequence.

    Supported layouts:
      1. <ROOT_2D>/<subject>/gt/<exercise>.json
      2. <VIDEOS_ROOT>/<subject>/<exercise>/joints3d_25.json (original Fit3D layout)
    """
    candidates = [
        os.path.join(ROOT_2D, subject, "gt", f"{video_name}.json"),
        os.path.join(ROOT_2D, subject, "gt (1)", f"{video_name}.json"),
        os.path.join(VIDEOS_ROOT, subject, video_name, "joints3d_25.json"),
    ]

    return next((path for path in candidates if os.path.isfile(path)), None)


def load_gt_3d(gt_path):
    """Load the 17-joint Fit3D ground truth, root-aligned and with the leg exchange."""
    with open(gt_path, "r", encoding="utf-8") as file:
        gt_json = json.load(file)

    if "joints3d_25" not in gt_json:
        raise KeyError(f"'joints3d_25' not found in {gt_path}.")

    gt = np.asarray(gt_json["joints3d_25"], dtype=np.float32)[:, :17, :]

    return swap_legs(root_align(gt))

## 6. Temporal Inference

TCPFormer receives windows of **243 consecutive frames**, whereas Fit3D sequences are much longer. Instead of processing independent, non-overlapping blocks, which creates discontinuities at the block boundaries, complete sequences are reconstructed with **overlapping windows fused with Hann weighting**:

1. **Window generation.** Windows of 243 frames start every 60 frames, so consecutive windows overlap by 183 frames and every interior frame is predicted by about four windows.
2. **Full coverage.** The last window is anchored to the end of the sequence, so every frame is covered.
3. **Short sequences.** A sequence with fewer than 243 frames is padded by repeating its last frame; the padded predictions are discarded afterwards.
4. **Batched inference.** `OVERLAP_BATCH_SIZE` windows are processed simultaneously. This only affects speed and memory.
5. **Hann-weighted fusion.** The predictions of each frame are averaged with a Hann weighting that favours the centre of each window, where the temporal context is richest. The weights are strictly positive (the two zero-valued end points of the Hann window are discarded), so no frame has zero weight:

$$
\hat{\mathbf{P}}_t
=
\frac{\sum_{k} w\!\left(t - s_k\right)\,\hat{\mathbf{P}}^{(k)}_{t - s_k}}
{\sum_{k} w\!\left(t - s_k\right)}
$$

where $s_k$ is the first frame of window $k$, $\hat{\mathbf{P}}^{(k)}$ its prediction and $w(\cdot)$ the Hann weight.

The pelvis root-centering is applied **after** the fusion, and no left/right exchange is applied to the output.

In [ ]:
# Overlapping-window inference with Hann-weighted fusion

def window_starts(num_frames, clip_len, stride):
    """First frame of every window; the last window is anchored to the end."""
    starts = list(range(0, num_frames - clip_len + 1, stride))

    if starts[-1] != num_frames - clip_len:
        starts.append(num_frames - clip_len)

    return starts


def unpack_model_output(output):
    """Extract the prediction tensor from the model output."""
    if isinstance(output, torch.Tensor):
        return output

    if isinstance(output, (list, tuple)):
        output = {index: item for index, item in enumerate(output)}

    if isinstance(output, dict):
        for key in ["pred", "pred_3d", "output", "outputs"]:
            if isinstance(output.get(key), torch.Tensor):
                return output[key]

        for value in output.values():
            if isinstance(value, torch.Tensor):
                return value

    raise TypeError(f"Unsupported model output type: {type(output)}")


def infer_overlap_hann(model, keypoints, clip_len=CLIP_LEN, stride=OVERLAP_STRIDE, batch_size=OVERLAP_BATCH_SIZE):
    """Predict a whole sequence with overlapping windows fused by Hann weighting.

    Parameters
    ----------
    model : torch.nn.Module
        Pretrained TCPFormer in evaluation mode.
    keypoints : np.ndarray
        Normalized 2D input of shape (T, 17, 3).

    Returns
    -------
    np.ndarray
        3D prediction of shape (T, 17, 3), not yet root-aligned.
    """
    sequence = np.asarray(keypoints, dtype=np.float32)
    original_length = len(sequence)

    if sequence.ndim != 3 or sequence.shape[1] != 17 or original_length == 0:
        raise ValueError(f"Expected a non-empty input of shape (T, 17, C), received {sequence.shape}.")

    # Pad only when the whole sequence is shorter than one window.
    if original_length < clip_len:
        padding = np.repeat(sequence[-1:], clip_len - original_length, axis=0)
        sequence = np.concatenate([sequence, padding], axis=0)

    starts = window_starts(len(sequence), clip_len, stride)

    # Strictly positive Hann weights: np.hanning(n + 2) without its zero end points.
    weights = np.hanning(clip_len + 2)[1:-1].astype(np.float32)[:, None, None]

    accumulated = np.zeros((len(sequence), 17, 3), dtype=np.float32)
    weight_sum = np.zeros((len(sequence), 1, 1), dtype=np.float32)

    with torch.no_grad():
        for begin in range(0, len(starts), batch_size):
            batch_starts = starts[begin:begin + batch_size]
            batch = np.stack([sequence[start:start + clip_len] for start in batch_starts])

            output = unpack_model_output(model(torch.from_numpy(batch).float().to(device)))
            output = output.detach().float().cpu().numpy()

            if output.shape[1:] != (clip_len, 17, 3):
                raise ValueError(f"Unexpected model output shape {output.shape}; expected (B, {clip_len}, 17, 3).")

            for start, window_prediction in zip(batch_starts, output):
                accumulated[start:start + clip_len] += weights * window_prediction
                weight_sum[start:start + clip_len] += weights

    if np.any(weight_sum <= 0):
        raise RuntimeError("Some frames received no prediction.")

    return (accumulated / weight_sum)[:original_length].astype(np.float32)

## 7. Evaluation Metrics

The predictions are evaluated with two standard spatial accuracy metrics. Both are computed on pelvis-centred poses and reported in millimetres.

**MPJPE** (Mean Per Joint Position Error) is the mean Euclidean distance between predicted and ground-truth joints:

$$
\mathrm{MPJPE}
=
\frac{1}{TJ}
\sum_{t=1}^{T}
\sum_{j=1}^{J}
\left\|
\hat{\mathbf{P}}_{t,j}
-
\mathbf{P}_{t,j}
\right\|_2
$$

**PA-MPJPE** (Procrustes-Aligned MPJPE) first aligns every predicted pose to the ground truth with the similarity transformation (translation, rotation and uniform scale) that minimizes the error, and then computes the MPJPE. It measures the shape of the pose independently of the global orientation and scale, and is the main spatial metric of the paper. The mean PA-MPJPE of each of the 17 joints is also stored.

The temporal metric (MPJVE) is evaluated in `04_temporal_smoothing.ipynb`.

In [ ]:
# Evaluation metrics

def procrustes_align(prediction, ground_truth):
    """Align one predicted pose (J, 3) to the ground truth with a similarity transform."""
    mean_prediction = prediction.mean(axis=0)
    mean_ground_truth = ground_truth.mean(axis=0)

    prediction_centered = prediction - mean_prediction
    ground_truth_centered = ground_truth - mean_ground_truth

    norm_prediction = np.linalg.norm(prediction_centered)
    norm_ground_truth = np.linalg.norm(ground_truth_centered)

    U, singular_values, Vt = np.linalg.svd(
        (prediction_centered / norm_prediction).T @ (ground_truth_centered / norm_ground_truth)
    )

    rotation = Vt.T @ U.T

    if np.linalg.det(rotation) < 0:
        Vt[-1] *= -1
        rotation = Vt.T @ U.T

    scale = norm_ground_truth / norm_prediction * np.sum(singular_values)
    translation = mean_ground_truth - scale * (rotation @ mean_prediction)

    return (scale * (rotation @ prediction.T)).T + translation


def evaluate_metrics(prediction, ground_truth):
    """Compute MPJPE, PA-MPJPE and per-joint PA-MPJPE (all in millimetres)."""
    aligned = np.stack([procrustes_align(p, g) for p, g in zip(prediction, ground_truth)])

    error = np.linalg.norm(prediction - ground_truth, axis=-1)
    error_pa = np.linalg.norm(aligned - ground_truth, axis=-1)

    return {
        "MPJPE_mm": float(error.mean() * 1000.0),
        "PA_MPJPE_mm": float(error_pa.mean() * 1000.0),
        "PerJoint_PA_MPJPE_mm": error_pa.mean(axis=0) * 1000.0,
    }

## 8. Running the Experiment

The experiment is run independently for every sequence of the selected 2D source. For each sequence:

1. Load the 2D keypoints and the image size, and apply the leg exchange if the source requires it.
2. Normalize the coordinates and run the overlapping-window inference (Section 6).
3. Root-align the prediction on the pelvis.
4. Load the 3D ground truth (Section 5.3) and truncate prediction and ground truth to their common length.
5. Compute MPJPE, PA-MPJPE and the per-joint PA-MPJPE (Section 7).
6. Save the prediction and append the results to the CSV files.

The run is **resumable**: a sequence whose prediction file and CSV entry already exist is skipped. A sequence that fails is reported and the run continues with the next one.

**Output files.** The names include the 2D source and the inference configuration (`<source>_<RUN_TAG>`, e.g. `pose2d_rtmpose_overlap_hann_w243_s60`), so results obtained with a different source or windowing never overwrite each other:

| File | Content |
|---|---|
| `pred3d_<source>_<RUN_TAG>/<subject>/<exercise>.npy` | 3D predictions `(T, 17, 3)`, pelvis-centred, in metres |
| `results_per_video_<source>_<RUN_TAG>.csv` | `subject`, `video`, `pose_source`, `mode`, `frames`, `MPJPE_mm`, `PA_MPJPE_mm`, `time_sec` |
| `results_per_joint_<source>_<RUN_TAG>.csv` | `subject`, `video`, `pose_source`, `joint`, `PA_MPJPE_mm` for each of the 17 joints |

In [ ]:
# Output locations and result bookkeeping

PRED_DIR = os.path.join(SAVE_ROOT, f"pred3d_{POSE_SOURCE}_{RUN_TAG}")
PER_VIDEO_CSV = os.path.join(SAVE_ROOT, f"results_per_video_{POSE_SOURCE}_{RUN_TAG}.csv")
PER_JOINT_CSV = os.path.join(SAVE_ROOT, f"results_per_joint_{POSE_SOURCE}_{RUN_TAG}.csv")

os.makedirs(PRED_DIR, exist_ok=True)


def append_rows(path, rows):
    """Append rows to a CSV file, writing the header when the file is created."""
    pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def load_completed_sequences(csv_path):
    """Return the (subject, video) pairs already registered in the per-video CSV."""
    if not os.path.exists(csv_path):
        return set()

    try:
        table = pd.read_csv(csv_path)
        return set(zip(table["subject"].astype(str), table["video"].astype(str)))
    except Exception:
        return set()


def collect_sequences():
    """List the sequences to process as (subject, video, pose_dir, gt_path)."""
    subjects = sorted(
        name for name in os.listdir(ROOT_2D) if os.path.isdir(os.path.join(ROOT_2D, name))
    )

    if SUBJECTS is not None:
        subjects = [subject for subject in subjects if subject in SUBJECTS]

    sequences = []

    for subject in subjects:
        pose_dir = resolve_pose_dir(subject)

        if pose_dir is None:
            print(f"SKIP | {subject} | folder of the 2D source '{POSE_SOURCE}' not found.")
            continue

        videos = list_pose_videos(pose_dir)

        if MAX_SEQUENCES_PER_SUBJECT is not None:
            videos = videos[:MAX_SEQUENCES_PER_SUBJECT]

        for video in videos:
            gt_path = find_gt_path(subject, video)

            if gt_path is None:
                print(f"SKIP | {subject}/{video} | ground truth not found.")
                continue

            sequences.append((subject, video, pose_dir, gt_path))

    return sequences


SEQUENCES = collect_sequences()

print(f"2D source: {POSE_SOURCE} | configuration: {RUN_TAG}")
print(f"Sequences to process: {len(SEQUENCES)} from {len({s[0] for s in SEQUENCES})} subjects")
print(f"Predictions: {PRED_DIR}")

In [ ]:
# Run the TCPFormer inference and evaluation

def process_sequence(subject, video_name, pose_dir, gt_path):
    """Predict and evaluate one sequence; return the prediction and its metrics."""
    keypoints, image_size = load_2d_keypoints(subject, video_name, pose_dir)

    if SOURCE["swap_legs"]:
        keypoints = swap_legs(keypoints)

    normalized = normalize_keypoints(keypoints, image_size["width"], image_size["height"])

    prediction = root_align(infer_overlap_hann(model, normalized))
    ground_truth = load_gt_3d(gt_path)

    num_frames = min(len(prediction), len(ground_truth))
    prediction, ground_truth = prediction[:num_frames], ground_truth[:num_frames]

    return prediction, evaluate_metrics(prediction, ground_truth)


completed = load_completed_sequences(PER_VIDEO_CSV)
failed = []

for subject, video_name, pose_dir, gt_path in SEQUENCES:
    pred_path = os.path.join(PRED_DIR, subject, f"{video_name}.npy")

    if (subject, video_name) in completed and os.path.exists(pred_path):
        print(f"SKIP | {subject}/{video_name} | already processed")
        continue

    start_time = time.time()

    try:
        prediction, metrics = process_sequence(subject, video_name, pose_dir, gt_path)

        os.makedirs(os.path.dirname(pred_path), exist_ok=True)
        np.save(pred_path, prediction)

        elapsed = time.time() - start_time

        append_rows(PER_VIDEO_CSV, [{
            "subject": subject,
            "video": video_name,
            "pose_source": POSE_SOURCE,
            "mode": RUN_TAG,
            "frames": len(prediction),
            "MPJPE_mm": metrics["MPJPE_mm"],
            "PA_MPJPE_mm": metrics["PA_MPJPE_mm"],
            "time_sec": elapsed,
        }])

        append_rows(PER_JOINT_CSV, [
            {
                "subject": subject,
                "video": video_name,
                "pose_source": POSE_SOURCE,
                "joint": joint_name,
                "PA_MPJPE_mm": float(error),
            }
            for joint_name, error in zip(JOINT_NAMES, metrics["PerJoint_PA_MPJPE_mm"])
        ])

        print(
            f"{subject}/{video_name} | MPJPE={metrics['MPJPE_mm']:.1f} mm | "
            f"PA-MPJPE={metrics['PA_MPJPE_mm']:.1f} mm | {elapsed:.1f} s"
        )

    except Exception as error:
        failed.append((subject, video_name, str(error)))
        print(f"ERROR | {subject}/{video_name} | {error}")

    if device == "cuda":
        torch.cuda.empty_cache()

print(f"\nFinished: {len(SEQUENCES) - len(failed)}/{len(SEQUENCES)} sequences without errors.")

if failed:
    print("Sequences with errors (re-run this cell to retry them):")
    for subject, video_name, message in failed:
        print(f"  {subject}/{video_name}: {message}")

## 9. Output Verification

The next cell checks the integrity of the files generated for the selected 2D source:

- **Predictions:** every expected file exists, can be loaded, has shape `(T, 17, 3)`, contains no NaN/Inf values and is pelvis-centred; there are no unexpected files.
- **Frame counts:** the number of frames equals the shorter of the 2D input and the 3D ground truth (`CHECK_FRAME_COUNTS`; it re-reads the inputs, so it is slower).
- **Per-video CSV:** required columns, no duplicates, one row per sequence, the expected inference configuration, finite values and a frame count consistent with the prediction file.
- **Per-joint CSV:** required columns, no duplicates, 17 joints per sequence and finite values.

The verdict is `OK` when no problem is found.

In [ ]:
# Verify the generated outputs

CHECK_FRAME_COUNTS = True   # re-reads the 2D inputs and the ground truth (slower)


def verify_outputs(sequences, check_frame_counts=True):
    """Check the generated files; return the list of problems and the number of valid predictions."""
    problems = []

    def report(subject, video, message):
        problems.append({"subject": subject, "video": video, "problem": message})

    expected = {(subject, video): (pose_dir, gt_path) for subject, video, pose_dir, gt_path in sequences}
    prediction_frames = {}
    valid_files = 0

    # 1. Prediction files
    for (subject, video), (pose_dir, gt_path) in sorted(expected.items()):
        path = os.path.join(PRED_DIR, subject, f"{video}.npy")

        if not os.path.exists(path):
            report(subject, video, "Missing prediction file")
            continue

        try:
            prediction = np.load(path)
        except Exception as error:
            report(subject, video, f"Prediction cannot be loaded: {error}")
            continue

        if prediction.ndim != 3 or prediction.shape[1:] != (17, 3):
            report(subject, video, f"Unexpected shape {prediction.shape}")
            continue

        if not np.isfinite(prediction).all():
            report(subject, video, "NaN or Inf values in prediction")
            continue

        if not np.allclose(prediction[:, 0], 0.0, atol=1e-5):
            report(subject, video, "Prediction is not pelvis-centred")
            continue

        prediction_frames[(subject, video)] = len(prediction)

        if check_frame_counts:
            keypoints, _ = load_2d_keypoints(subject, video, pose_dir)
            expected_frames = min(len(keypoints), len(load_gt_3d(gt_path)))

            if len(prediction) != expected_frames:
                report(subject, video, f"Frames: prediction={len(prediction)}, expected={expected_frames}")
                continue

        valid_files += 1

    if os.path.isdir(PRED_DIR):
        for subject in sorted(os.listdir(PRED_DIR)):
            subject_dir = os.path.join(PRED_DIR, subject)

            if os.path.isdir(subject_dir):
                for name in sorted(os.listdir(subject_dir)):
                    if name.endswith(".npy") and (subject, name[:-4]) not in expected:
                        report(subject, name[:-4], "Unexpected prediction file")

    # 2. Per-video CSV
    if not os.path.exists(PER_VIDEO_CSV):
        report("", "", "Per-video CSV not found")
    else:
        table = pd.read_csv(PER_VIDEO_CSV)
        required = {"subject", "video", "pose_source", "mode", "frames", "MPJPE_mm", "PA_MPJPE_mm", "time_sec"}

        if required - set(table.columns):
            report("", "", f"Per-video CSV lacks columns {sorted(required - set(table.columns))}")
        else:
            if table.duplicated(subset=["subject", "video"]).any():
                report("", "", "Duplicated rows in per-video CSV")

            rows = set(zip(table["subject"].astype(str), table["video"].astype(str)))

            for subject, video in sorted(set(expected) - rows):
                report(subject, video, "Missing row in per-video CSV")

            for subject, video in sorted(rows - set(expected)):
                report(subject, video, "Unexpected row in per-video CSV")

            for _, row in table[table["mode"].astype(str) != RUN_TAG].iterrows():
                report(row["subject"], row["video"], f"Unexpected mode: {row['mode']}")

            for column in ["MPJPE_mm", "PA_MPJPE_mm", "time_sec"]:
                invalid = int((~np.isfinite(pd.to_numeric(table[column], errors="coerce"))).sum())

                if invalid:
                    report("", "", f"{invalid} invalid values in {column}")

            for _, row in table.iterrows():
                pair = (str(row["subject"]), str(row["video"]))

                if pair in prediction_frames and int(row["frames"]) != prediction_frames[pair]:
                    report(pair[0], pair[1], f"CSV frames={int(row['frames'])}, prediction frames={prediction_frames[pair]}")

    # 3. Per-joint CSV
    if not os.path.exists(PER_JOINT_CSV):
        report("", "", "Per-joint CSV not found")
    else:
        table = pd.read_csv(PER_JOINT_CSV)
        required = {"subject", "video", "pose_source", "joint", "PA_MPJPE_mm"}

        if required - set(table.columns):
            report("", "", f"Per-joint CSV lacks columns {sorted(required - set(table.columns))}")
        else:
            if table.duplicated(subset=["subject", "video", "joint"]).any():
                report("", "", "Duplicated rows in per-joint CSV")

            unknown_joints = sorted(set(table["joint"].astype(str)) - set(JOINT_NAMES))

            if unknown_joints:
                report("", "", f"Per-joint CSV contains joint names that are not in JOINT_NAMES: {unknown_joints}. "
                               "It was probably produced by an older version of the notebook: move the output folder and run again.")

            joints_per_video = table.groupby(["subject", "video"])["joint"].nunique()

            for (subject, video), count in joints_per_video.items():
                if count != len(JOINT_NAMES):
                    report(subject, video, f"{count} joints in per-joint CSV (expected {len(JOINT_NAMES)})")

            for subject, video in sorted(set(expected) - set(joints_per_video.index)):
                report(subject, video, "Missing sequence in per-joint CSV")

            invalid = int((~np.isfinite(pd.to_numeric(table["PA_MPJPE_mm"], errors="coerce"))).sum())

            if invalid:
                report("", "", f"{invalid} invalid per-joint PA-MPJPE values")

    return problems, valid_files


problems, valid_files = verify_outputs(SEQUENCES, CHECK_FRAME_COUNTS)

# The complete experiment must contain 8 subjects x 28 exercises (224 sequences)
if not IS_SUBSET:
    sequences_per_subject = pd.Series([sequence[0] for sequence in SEQUENCES]).value_counts()

    if len(SEQUENCES) != 224 or len(sequences_per_subject) != 8 or not (sequences_per_subject == 28).all():
        problems.append({
            "subject": "", "video": "",
            "problem": f"Incomplete experiment: {len(SEQUENCES)} sequences found (expected 224 = 8 subjects x 28); "
                       f"per subject: {sequences_per_subject.to_dict()}",
        })

print(f"Source: {POSE_SOURCE} | configuration: {RUN_TAG}")
print(f"Expected sequences: {len(SEQUENCES)}")
print(f"Valid prediction files: {valid_files}/{len(SEQUENCES)}")

if problems:
    print(f"\nProblems detected: {len(problems)}")
    print(pd.DataFrame(problems).head(50).to_string(index=False))
else:
    print("\nNo structural problems detected.")

print("\nVERDICT: " + ("OK" if not problems else "REVIEW THE PROBLEMS LISTED ABOVE"))

## 10. Summary

The last cell summarizes the run and, for the complete experiment, prints the PA-MPJPE (17 joints) reported in the paper for the same model and 2D source. The mean is given both over sequences and weighted by the number of frames. Small differences with respect to the paper may appear with a different GPU or library versions.

In [ ]:
# Summary of the run

PAPER_PA_MPJPE_MM = {   # PA-MPJPE (mm), 17 joints, TCPFormer, complete experiment
    "poses2d_gt": 46.16,
    "pose2d_yolo": 93.61,
    "pose2d_rtmpose": 70.89,
}

if not os.path.exists(PER_VIDEO_CSV):
    print("No results found: check the messages printed in Section 8.")
else:
    summary = pd.read_csv(PER_VIDEO_CSV)

    print(f"Model: TCPFormer | 2D source: {POSE_SOURCE} | configuration: {RUN_TAG}")
    print(f"Sequences evaluated: {len(summary)}")
    print(f"MPJPE (mean over sequences):    {summary['MPJPE_mm'].mean():.2f} mm")
    print(f"PA-MPJPE (mean over sequences): {summary['PA_MPJPE_mm'].mean():.2f} mm")
    print(f"PA-MPJPE (frame-weighted):      {np.average(summary['PA_MPJPE_mm'], weights=summary['frames']):.2f} mm")

    if IS_SUBSET:
        print("Subset run: the values are not comparable with the complete experiment of the paper.")
    else:
        print(f"PA-MPJPE reported in the paper: {PAPER_PA_MPJPE_MM[POSE_SOURCE]:.2f} mm (224 sequences expected)")

## Troubleshooting

| Symptom | Solution |
|---|---|
| `ERROR: pip's dependency resolver ...` when installing | Informative, see the note in Section 2. |
| `CUDA out of memory` | Reduce `OVERLAP_BATCH_SIZE` (for example to 4 or 2); the results do not change. |
| `TCPFormer checkpoint not found` | Place `TCPFormer_h36m_243_379.pth.tr` in `STORAGE_ROOT/checkpoints/`. |
| `WARNING ... no width/height metadata found` | Expected when only the RTMPose or ground-truth 2D files are available; the default 900 × 900 is used. |
| `SKIP | ... ground truth not found` | Check that `joints3d_25.json` exists for that sequence (see Section 1). |
| Results of a previous run are reused | Delete the output CSV files and prediction folder of that source to recompute everything. |